In [1]:
!unzip SumArabic.zip

Archive:  SumArabic.zip
  inflating: sumarabic-1.0-ood.jsonl  
  inflating: sumarabic-1.0-test.jsonl  
  inflating: sumarabic-1.0-train.jsonl  
  inflating: sumarabic-1.0-valid.jsonl  


In [2]:
import pandas as pd
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

# change paths
train_path = "sumarabic-1.0-train.jsonl"
valid_path = "sumarabic-1.0-valid.jsonl"
test_path = "sumarabic-1.0-test.jsonl"
ood_path   = "sumarabic-1.0-ood.jsonl"

df_train = load_jsonl(train_path)
df_valid = load_jsonl(valid_path)
df_test = load_jsonl(test_path)
df_ood = load_jsonl(ood_path)

for name, df_part in [("train", df_train), ("valid", df_valid), ("test", df_test), ("ood", df_ood)]:
    df_part["split"] = name

df = pd.concat([df_train, df_valid, df_test, df_ood], ignore_index=True)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df[["text", "headline", "section", "published", "split"]].head(3))

Shape: (84764, 11)
Columns: ['dataset', 'filename', 'headline', 'length', 'offset', 'published', 'section', 'subdomain', 'text', 'url', 'split']
                                                text  \
0  اختتمت مساء أول من أمس نهائيات بطولة الإمارات ...   
1  مينيابوليس (الولايات المتحدة) 3-2-2008 (ا ف ب)...   
2  أفاد مصدر في شرطة أبوظبي بأن نحو 700 طفل يتواف...   

                                     headline           section  \
0   المصري فؤاد الطاهر بطل للشطرنج الديناميكي  emaratalyoum.com   
1  حملة انتخابية محمومة قبل "الثلاثاء الكبير"  emaratalyoum.com   
2              55% من نزلاء «الأحداث» مواطنون  emaratalyoum.com   

               published  split  
0  2008-02-03 00:00+0400  train  
1  2008-02-03 00:00+0400  train  
2  2008-02-04 00:00+0400  train  


In [3]:
ARABIC_RE = re.compile(r'[\u0600-\u06FF]')
ENGLISH_RE = re.compile(r'[A-Za-z]')
HTML_TAG_RE = re.compile(r'<[^>]+>')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')
DIACRITICS_RE = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670]')
TATWEEL_RE = re.compile(r'ـ')
DIGIT_RE = re.compile(r'\d')
REPEATED_PUNCT_RE = re.compile(r'([!?؟،؛.,:])\1{1,}')
CONTROL_CHARS_RE = re.compile(r'[\x00-\x1F\x7F]')
INVISIBLE_UNICODE_RE = re.compile(r'[\u200b-\u200f\u202a-\u202e\u2066-\u2069]')

def inspect_text(text):
    text = "" if pd.isna(text) else str(text)

    total_chars = len(text)
    word_count = len(text.split())

    arabic_chars = len(ARABIC_RE.findall(text))
    english_chars = len(ENGLISH_RE.findall(text))
    digit_chars = len(DIGIT_RE.findall(text))

    suspicious_chars = []
    for ch in text:
        if ch in "\n\r\t ":
            continue
        if ARABIC_RE.match(ch) or ENGLISH_RE.match(ch) or ch.isdigit():
            continue
        if ch in ".,!?؟،؛:()[]{}\"'«»-_/\\%+*=|<>":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C") or cat.startswith("S"):
            suspicious_chars.append(ch)

    return {
        "char_len": total_chars,
        "word_count": word_count,
        "arabic_ratio": arabic_chars / total_chars if total_chars else 0,
        "english_chars": english_chars,
        "digit_chars": digit_chars,
        "has_html": bool(HTML_TAG_RE.search(text)),
        "has_url": bool(URL_RE.search(text)),
        "has_email": bool(EMAIL_RE.search(text)),
        "has_diacritics": bool(DIACRITICS_RE.search(text)),
        "has_tatweel": bool(TATWEEL_RE.search(text)),
        "has_repeated_punct": bool(REPEATED_PUNCT_RE.search(text)),
        "has_control_chars": bool(CONTROL_CHARS_RE.search(text)),
        "has_invisible_unicode": bool(INVISIBLE_UNICODE_RE.search(text)),
        "suspicious_char_count": len(suspicious_chars),
        "suspicious_chars_sample": "".join(sorted(set(suspicious_chars)))[:50],
    }

In [4]:
text_diag = df["text"].fillna("").apply(inspect_text).apply(pd.Series).add_prefix("text_")
headline_diag = df["headline"].fillna("").apply(inspect_text).apply(pd.Series).add_prefix("headline_")

scan_df = pd.concat([df, text_diag, headline_diag], axis=1)

scan_df["text_headline_same"] = (
    scan_df["text"].fillna("").str.strip() == scan_df["headline"].fillna("").str.strip()
)

scan_df["headline_longer_than_text"] = (
    scan_df["headline_word_count"] > scan_df["text_word_count"]
)

scan_df["very_short_text"] = scan_df["text_word_count"] < 15
scan_df["very_short_headline"] = scan_df["headline_word_count"] < 3

In [5]:
for split in ["train", "valid", "test", "ood"]:
    part = scan_df[scan_df["split"] == split]
    print(f"\n=== {split.upper()} ===")
    print("rows:", len(part))
    print("empty text:", (part["text"].str.strip() == "").sum())
    print("empty headline:", (part["headline"].str.strip() == "").sum())
    print("duplicate pairs:", part.duplicated(subset=["text", "headline"]).sum())
    print("text urls:", part["text_has_url"].sum())
    print("headline urls:", part["headline_has_url"].sum())
    print("text english:", (part["text_english_chars"] > 0).sum())
    print("headline english:", (part["headline_english_chars"] > 0).sum())
    print("text digits:", (part["text_digit_chars"] > 0).sum())
    print("headline digits:", (part["headline_digit_chars"] > 0).sum())
    print("very short text:", part["very_short_text"].sum())
    print("headline longer than text:", part["headline_longer_than_text"].sum())


=== TRAIN ===
rows: 75817
empty text: 0
empty headline: 0
duplicate pairs: 0
text urls: 0
headline urls: 0
text english: 2122
headline english: 439
text digits: 34638
headline digits: 22083
very short text: 1087
headline longer than text: 0

=== VALID ===
rows: 4121
empty text: 0
empty headline: 0
duplicate pairs: 0
text urls: 0
headline urls: 0
text english: 90
headline english: 15
text digits: 1899
headline digits: 1231
very short text: 56
headline longer than text: 0

=== TEST ===
rows: 4174
empty text: 0
empty headline: 0
duplicate pairs: 0
text urls: 0
headline urls: 0
text english: 116
headline english: 20
text digits: 1889
headline digits: 1194
very short text: 58
headline longer than text: 0

=== OOD ===
rows: 652
empty text: 0
empty headline: 0
duplicate pairs: 0
text urls: 0
headline urls: 0
text english: 2
headline english: 0
text digits: 281
headline digits: 140
very short text: 10
headline longer than text: 0


In [6]:
print("\nText length stats:")
print(scan_df["text_word_count"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\nHeadline length stats:")
print(scan_df["headline_word_count"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))


Text length stats:
count    84764.000000
mean        33.618612
std         10.961317
min         10.000000
50%         32.000000
90%         50.000000
95%         55.000000
99%         59.000000
max         60.000000
Name: text_word_count, dtype: float64

Headline length stats:
count    84764.000000
mean         8.403969
std          2.051067
min          3.000000
50%          8.000000
90%         11.000000
95%         12.000000
99%         14.000000
max         15.000000
Name: headline_word_count, dtype: float64


In [7]:
def show_examples(df_, condition, cols=None, n=5, title="Examples", max_chars=1000):
    cols = cols or ["text", "headline", "section", "url"]
    sample = df_.loc[condition, cols].head(n)
    print(f"\n===== {title} =====")
    if sample.empty:
        print("No examples found.")
        return
    for i, row in sample.iterrows():
        print(f"\n--- Row {i} ---")
        for c in cols:
            print(f"\n[{c}]")
            print(str(row[c])[:max_chars])

show_examples(scan_df, scan_df["very_short_text"], title="Very short texts")
show_examples(scan_df, scan_df["headline_longer_than_text"], title="Headline longer than text")
show_examples(scan_df, scan_df["text_english_chars"] > 0, title="English in text")
show_examples(scan_df, scan_df["text_has_url"], title="URLs in text")


===== Very short texts =====

--- Row 1 ---

[text]
مينيابوليس (الولايات المتحدة) 3-2-2008 (ا ف ب) - توقع المرشح الجمهوري جون

[headline]
حملة انتخابية محمومة قبل "الثلاثاء الكبير"

[section]
emaratalyoum.com

[url]
https://www.emaratalyoum.com/local-section/2008-02-03-1.190756

--- Row 50 ---

[text]
أعربت دولة الإمارات عن رغبتها في اقامة شراكات تعليمية مع الجمهورية الألمانية الاتحادية.

[headline]
محمد بن راشد يدعو إلى شراكات تعليمية مع ألمانيا

[section]
emaratalyoum.com

[url]
https://www.emaratalyoum.com/local-section/2008-02-08-1.191516

--- Row 77 ---

[text]
أشعلت التعديلات الجديدة التي لحقت ببطولة سمو الشيخ حمدان بن محمد بن

[headline]
تعديلات «فزاع» للرماية تشعل المنافسة

[section]
emaratalyoum.com

[url]
https://www.emaratalyoum.com/local-section/2008-02-09-1.190517

--- Row 499 ---

[text]
أكد باحثون أميركيون «أن تناول الفيتامينات المتعددة والمواد المعدنية يمكن أن يمنع زحف الشيخوخة». 

[headline]
نقص الفيتامينات يهدد سلامة الجسم

[section]
emaratalyoum.com

[url]
https://w

In [8]:
def clean_sumarabic(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()

    # Remove diacritics + tatweel
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652\u0670\u0640]', '', text)

    # Light Arabic normalization
    text = re.sub(r'[إأآ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
df = df[["text", "headline", "section", "published", "url", "split"]].copy()

df = df.dropna(subset=["text", "headline"]).drop_duplicates(subset=["text", "headline"]).reset_index(drop=True)

In [10]:
df["text"] = df["text"].apply(clean_sumarabic)
df["headline"] = df["headline"].apply(clean_sumarabic)

In [11]:
df = df[
    (df["text"].str.strip() != "") &
    (df["headline"].str.strip() != "")
].copy()

df = df[df["text"].str.split().str.len() >= 12].copy()

df["headline"] = df["headline"].apply(lambda x: f"<start> {x} <end>")

df.to_csv("cleaned_sumarabic_final.csv", index=False, encoding="utf-8-sig")